<a href="https://colab.research.google.com/github/haleshanov-create/Digital-Harassment-KZ./blob/main/Digital_Harassment_KZ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Digital Harassment KZ

Ячейка 1 — установка библиотек

In [1]:
%pip -q install requests beautifulsoup4 pandas feedparser trafilatura plotly gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.7/254.7 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 22.7 MB/s eta 0:00:00


Ячейка 2 — источники и ключевые слова

In [2]:
import requests
import feedparser
import pandas as pd
import re
import time
from urllib.parse import quote

SEARCH_QUERIES = [
    'site:tengrinews.kz шантаж OR вымогательство',
    'site:tengrinews.kz кибербуллинг OR преследование',
    'site:inform.kz шантаж OR интернет-угрозы',
    'site:inform.kz кибербуллинг OR цифровое насилие',
    'site:orda.kz шантаж OR преследование',
    'site:zakon.kz шантаж OR вымогательство',
    'site:baigenews.kz кибербуллинг OR шантаж',
    'Қазақстан бопсалау интернет қудалау',
    'Қазақстан кибербуллинг қорқыту'
]

RSS_TEMPLATE = (
    "https://news.google.com/rss/search?q={query}"
    "&hl=ru&gl=KZ&ceid=KZ:ru"
)

headers = {
    "User-Agent": "DigitalHarassmentKZ/1.0 (research project)"
}

print("Источники и поисковые запросы готовы:", len(SEARCH_QUERIES))

Источники и поисковые запросы готовы: 9


Ячейка 3 — сбор заголовков и ссылок

In [3]:
records = []

for query in SEARCH_QUERIES:
    rss_url = RSS_TEMPLATE.format(query=quote(query))

    try:
        response = requests.get(
            rss_url,
            headers=headers,
            timeout=20
        )
        response.raise_for_status()

        feed = feedparser.parse(response.content)

        for entry in feed.entries:
            title = entry.get("title", "").strip()
            link = entry.get("link", "").strip()
            published = entry.get("published", "")
            summary = re.sub(
                r"<[^>]+>",
                " ",
                entry.get("summary", "")
            ).strip()

            if not title or not link:
                continue

            records.append({
                "title": title,
                "date": published,
                "summary": summary,
                "url": link,
                "search_query": query,
                "source": entry.get("source", {}).get("title", "")
            })

        print(f"Обработан запрос: {query}")

    except Exception as e:
        print(f"Не удалось обработать запрос: {query}")
        print(type(e).__name__, str(e)[:200])

    time.sleep(1)

news = pd.DataFrame(records)

if not news.empty:
    news = news.drop_duplicates(subset=["url"]).reset_index(drop=True)

print("Всего уникальных материалов:", len(news))
display(news.head(10))

Обработан запрос: site:tengrinews.kz шантаж OR вымогательство
Обработан запрос: site:tengrinews.kz кибербуллинг OR преследование
Обработан запрос: site:inform.kz шантаж OR интернет-угрозы
Обработан запрос: site:inform.kz кибербуллинг OR цифровое насилие
Обработан запрос: site:orda.kz шантаж OR преследование
Обработан запрос: site:zakon.kz шантаж OR вымогательство
Обработан запрос: site:baigenews.kz кибербуллинг OR шантаж
Обработан запрос: Қазақстан бопсалау интернет қудалау
Обработан запрос: Қазақстан кибербуллинг қорқыту
Всего уникальных материалов: 577


,title,date,summary,url,search_query,source
0,Астанчане шли на свидания и оставались без маш...,"Fri, 09 Oct 2026 07:59:37 GMT",Астанчане шли на свидания и оставались без маш...,https://news.google.com/rss/articles/CBMiogFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews
1,Вымогатели собирали “дань“ с водителей - МВД и...,"Thu, 24 Sep 2026 07:00:00 GMT",Вымогатели собирали “дань“ с водителей - МВД и...,https://news.google.com/rss/articles/CBMiogFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews
2,Казахстанец “оценил“ интимные видео бывшей в 7...,"Thu, 15 Oct 2026 08:35:14 GMT",Казахстанец “оценил“ интимные видео бывшей в 7...,https://news.google.com/rss/articles/CBMirAFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews
3,Отец Алсу подозревается в вымогательстве двух ...,"Wed, 22 Jan 2014 08:00:00 GMT",Отец Алсу подозревается в вымогательстве двух ...,https://news.google.com/rss/articles/CBMikwFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews
4,Ошибки в Damumed могут привести к шантажу и вы...,"Thu, 03 Sep 2020 07:00:00 GMT",Ошибки в Damumed могут привести к шантажу и вы...,https://news.google.com/rss/articles/CBMiowFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews
5,Избившего таксиста дипломата заподозрили в вым...,"Tue, 09 Aug 2011 07:00:00 GMT",Избившего таксиста дипломата заподозрили в вым...,https://news.google.com/rss/articles/CBMipAFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews
6,"В Алматы задержан лидер ОПГ ""Атаба"" за вымогат...","Mon, 24 Jan 2011 08:00:00 GMT","В Алматы задержан лидер ОПГ ""Атаба"" за вымогат...",https://news.google.com/rss/articles/CBMilwFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews
7,Руководитель облздрава Мангистауской области п...,"Wed, 10 May 2017 07:00:00 GMT",Руководитель облздрава Мангистауской области п...,https://news.google.com/rss/articles/CBMimgFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews
8,Крупную схему вымогательства денег у туристов ...,"Mon, 02 Oct 2023 07:00:00 GMT",Крупную схему вымогательства денег у туристов ...,https://news.google.com/rss/articles/CBMiowFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews
9,Ерлан Карин об инфо-рэкете: мы не должны поощр...,"Thu, 27 Nov 2025 08:00:00 GMT",Ерлан Карин об инфо-рэкете: мы не должны поощр...,https://news.google.com/rss/articles/CBMioAFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews


Ячейка 4 — фильтрация по теме

In [4]:
KEYWORDS = {
    "Шантаж и вымогательство": [
        "шантаж", "вымогательств", "бопсалау", "қорқытып алу"
    ],
    "Цифровое преследование": [
        "цифровое преследование", "киберсталкинг",
        "сталкинг", "онлайн-преследование", "интернетте қудалау"
    ],
    "Кибербуллинг и угрозы": [
        "кибербуллинг", "онлайн-угроз", "угрозы в интернете",
        "буллинг", "қорқыту", "кибербуллинг"
    ],
    "Интимные материалы без согласия": [
        "слив интимных фото", "интимные фотографии без согласия",
        "интимдік суреттер", "жеке фотоларды тарату"
    ],
    "Цифровое мошенничество": [
        "взлом аккаунта", "шантаж в сети",
        "мошенничество в интернете", "аккаунтын бұзу"
    ]
}

def classify_text(text):
    text = str(text).lower()
    matches = []

    for category, words in KEYWORDS.items():
        if any(word in text for word in words):
            matches.append(category)

    return ", ".join(matches) if matches else "Требует проверки"

if not news.empty:
    news["category"] = (
        news["title"].fillna("") + " " +
        news["summary"].fillna("")
    ).apply(classify_text)

    relevant = news[
        news["category"] != "Требует проверки"
    ].copy()

    print("Всего найдено:", len(news))
    print("Предварительно релевантных:", len(relevant))
    display(relevant.head(20))
else:
    relevant = pd.DataFrame()
    print("Записи не получены. Проверь доступ к RSS и повтори позже.")

Всего найдено: 577
Предварительно релевантных: 150


,title,date,summary,url,search_query,source,category
3,Отец Алсу подозревается в вымогательстве двух ...,"Wed, 22 Jan 2014 08:00:00 GMT",Отец Алсу подозревается в вымогательстве двух ...,https://news.google.com/rss/articles/CBMikwFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews,Шантаж и вымогательство
4,Ошибки в Damumed могут привести к шантажу и вы...,"Thu, 03 Sep 2020 07:00:00 GMT",Ошибки в Damumed могут привести к шантажу и вы...,https://news.google.com/rss/articles/CBMiowFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews,Шантаж и вымогательство
5,Избившего таксиста дипломата заподозрили в вым...,"Tue, 09 Aug 2011 07:00:00 GMT",Избившего таксиста дипломата заподозрили в вым...,https://news.google.com/rss/articles/CBMipAFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews,Шантаж и вымогательство
6,"В Алматы задержан лидер ОПГ ""Атаба"" за вымогат...","Mon, 24 Jan 2011 08:00:00 GMT","В Алматы задержан лидер ОПГ ""Атаба"" за вымогат...",https://news.google.com/rss/articles/CBMilwFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews,Шантаж и вымогательство
7,Руководитель облздрава Мангистауской области п...,"Wed, 10 May 2017 07:00:00 GMT",Руководитель облздрава Мангистауской области п...,https://news.google.com/rss/articles/CBMimgFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews,Шантаж и вымогательство
8,Крупную схему вымогательства денег у туристов ...,"Mon, 02 Oct 2023 07:00:00 GMT",Крупную схему вымогательства денег у туристов ...,https://news.google.com/rss/articles/CBMiowFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews,Шантаж и вымогательство
9,Ерлан Карин об инфо-рэкете: мы не должны поощр...,"Thu, 27 Nov 2025 08:00:00 GMT",Ерлан Карин об инфо-рэкете: мы не должны поощр...,https://news.google.com/rss/articles/CBMioAFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews,Шантаж и вымогательство
10,Новости по теме шантаж - TengriNews,"Tue, 13 Feb 2024 13:06:21 GMT",Новости по теме шантаж &nbsp;&nbsp; TengriNews,https://news.google.com/rss/articles/CBMicEFVX...,site:tengrinews.kz шантаж OR вымогательство,TengriNews,Шантаж и вымогательство
11,Второй подсудимый по делу экс-главы АРЕМ расск...,"Mon, 26 Jan 2015 08:00:00 GMT",Второй подсудимый по делу экс-главы АРЕМ расск...,https://news.google.com/rss/articles/CBMinwFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews,Шантаж и вымогательство
12,Жалобы на насилие и вымогательство: управляюще...,"Thu, 29 Jan 2026 08:00:00 GMT",Жалобы на насилие и вымогательство: управляюще...,https://news.google.com/rss/articles/CBMimAFBV...,site:tengrinews.kz шантаж OR вымогательство,TengriNews,Шантаж и вымогательство


Ячейка 5 — сохранение датасета

In [5]:
output_path = "/content/digital_harassment_kz_news.csv"

relevant.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Файл сохранён:", output_path)

from google.colab import files
files.download(output_path)

Файл сохранён: /content/digital_harassment_kz_news.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Ячейка 6 — первая визуализация

In [6]:
import plotly.express as px

if not relevant.empty:
    category_counts = (
        relevant["category"]
        .value_counts()
        .rename_axis("Категория")
        .reset_index(name="Количество")
    )

    fig = px.bar(
        category_counts,
        x="Категория",
        y="Количество",
        title="Новости о цифровом преследовании и шантаже",
        text="Количество"
    )

    fig.update_layout(xaxis_tickangle=-25)
    fig.show()

    display(
        relevant[["date", "title", "category", "source", "url"]]
    )
else:
    print("Пока нет материалов для визуализации.")

,date,title,category,source,url
3,"Wed, 22 Jan 2014 08:00:00 GMT",Отец Алсу подозревается в вымогательстве двух ...,Шантаж и вымогательство,TengriNews,https://news.google.com/rss/articles/CBMikwFBV...
4,"Thu, 03 Sep 2020 07:00:00 GMT",Ошибки в Damumed могут привести к шантажу и вы...,Шантаж и вымогательство,TengriNews,https://news.google.com/rss/articles/CBMiowFBV...
5,"Tue, 09 Aug 2011 07:00:00 GMT",Избившего таксиста дипломата заподозрили в вым...,Шантаж и вымогательство,TengriNews,https://news.google.com/rss/articles/CBMipAFBV...
6,"Mon, 24 Jan 2011 08:00:00 GMT","В Алматы задержан лидер ОПГ ""Атаба"" за вымогат...",Шантаж и вымогательство,TengriNews,https://news.google.com/rss/articles/CBMilwFBV...
7,"Wed, 10 May 2017 07:00:00 GMT",Руководитель облздрава Мангистауской области п...,Шантаж и вымогательство,TengriNews,https://news.google.com/rss/articles/CBMimgFBV...
...,...,...,...,...,...
511,"Mon, 21 Aug 2017 07:00:00 GMT","В ЮКО экс-депутат по прозвищу ""Баха баха"" осуж...",Шантаж и вымогательство,Новости Zakon.kz,https://news.google.com/rss/articles/CBMilgFBV...
514,"Fri, 10 Jul 2026 07:00:00 GMT",Жительница Актобе шантажировала нового знакомо...,Шантаж и вымогательство,Новости Zakon.kz,https://news.google.com/rss/articles/CBMizgFBV...
526,"Wed, 18 Feb 2026 08:00:00 GMT",Зуб в торте или вымогательство: взаимные обвин...,Шантаж и вымогательство,Новости Zakon.kz,https://news.google.com/rss/articles/CBMixgFBV...
527,"Thu, 09 Apr 2026 07:00:00 GMT","Насилие, угрозы и шантаж: слухи раскрыли жутку...",Шантаж и вымогательство,Новости Zakon.kz,https://news.google.com/rss/articles/CBMi2AFBV...


Ячейка 7. Очистка и подготовка датасета

In [7]:
import pandas as pd
import re
from urllib.parse import urlparse

# Работаем с уже собранными материалами
df = relevant.copy()

# Гарантируем наличие нужных столбцов
for col in ["title", "date", "summary", "url", "source", "category"]:
    if col not in df.columns:
        df[col] = ""

# Очистка текстовых полей
for col in ["title", "summary", "url", "source", "category"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

# Оставляем только записи с заголовком и ссылкой
df = df[
    (df["title"] != "") &
    (df["url"].str.startswith("http"))
].copy()

# Удаляем повторяющиеся ссылки
df = df.drop_duplicates(subset=["url"])

# Преобразуем даты в единый формат
df["date_parsed"] = pd.to_datetime(
    df["date"],
    errors="coerce",
    utc=True
)

# Если название источника отсутствует, определяем домен из ссылки
def get_domain(url):
    try:
        return urlparse(url).netloc.lower().removeprefix("www.")
    except Exception:
        return ""

df["source"] = df.apply(
    lambda row: row["source"] if row["source"] else get_domain(row["url"]),
    axis=1
)

# Добавляем длину описания для контроля качества данных
df["summary_length"] = df["summary"].str.len()

# Сохраняем чистый датасет
df.to_csv(
    "/content/digital_harassment_kz_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("После очистки:", len(df), "материалов")
print("Дубликатов удалено:", len(relevant) - len(df))
print("Материалов с распознанной датой:", df["date_parsed"].notna().sum())

display(df[[
    "date", "title", "source", "category", "url"
]].head(10))

После очистки: 150 материалов
Дубликатов удалено: 0
Материалов с распознанной датой: 150


,date,title,source,category,url
3,"Wed, 22 Jan 2014 08:00:00 GMT",Отец Алсу подозревается в вымогательстве двух ...,TengriNews,Шантаж и вымогательство,https://news.google.com/rss/articles/CBMikwFBV...
4,"Thu, 03 Sep 2020 07:00:00 GMT",Ошибки в Damumed могут привести к шантажу и вы...,TengriNews,Шантаж и вымогательство,https://news.google.com/rss/articles/CBMiowFBV...
5,"Tue, 09 Aug 2011 07:00:00 GMT",Избившего таксиста дипломата заподозрили в вым...,TengriNews,Шантаж и вымогательство,https://news.google.com/rss/articles/CBMipAFBV...
6,"Mon, 24 Jan 2011 08:00:00 GMT","В Алматы задержан лидер ОПГ ""Атаба"" за вымогат...",TengriNews,Шантаж и вымогательство,https://news.google.com/rss/articles/CBMilwFBV...
7,"Wed, 10 May 2017 07:00:00 GMT",Руководитель облздрава Мангистауской области п...,TengriNews,Шантаж и вымогательство,https://news.google.com/rss/articles/CBMimgFBV...
8,"Mon, 02 Oct 2023 07:00:00 GMT",Крупную схему вымогательства денег у туристов ...,TengriNews,Шантаж и вымогательство,https://news.google.com/rss/articles/CBMiowFBV...
9,"Thu, 27 Nov 2025 08:00:00 GMT",Ерлан Карин об инфо-рэкете: мы не должны поощр...,TengriNews,Шантаж и вымогательство,https://news.google.com/rss/articles/CBMioAFBV...
10,"Tue, 13 Feb 2024 13:06:21 GMT",Новости по теме шантаж - TengriNews,TengriNews,Шантаж и вымогательство,https://news.google.com/rss/articles/CBMicEFVX...
11,"Mon, 26 Jan 2015 08:00:00 GMT",Второй подсудимый по делу экс-главы АРЕМ расск...,TengriNews,Шантаж и вымогательство,https://news.google.com/rss/articles/CBMinwFBV...
12,"Thu, 29 Jan 2026 08:00:00 GMT",Жалобы на насилие и вымогательство: управляюще...,TengriNews,Шантаж и вымогательство,https://news.google.com/rss/articles/CBMimAFBV...


Ячейка 8. Извлечение полного текста статей

In [8]:
import requests
import trafilatura
import time
from urllib.parse import urlparse

headers = {
    "User-Agent": "DigitalHarassmentKZ/1.0 (educational research)"
}

# Ограничиваем число запросов за один запуск
MAX_ARTICLES = min(len(df), 60)

article_texts = []
fetch_statuses = []

for i, row in enumerate(df.head(MAX_ARTICLES).itertuples(), start=1):
    url = row.url
    text = ""
    status = "Не удалось получить текст"

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=20
        )

        if response.status_code == 200:
            extracted = trafilatura.extract(
                response.text,
                include_comments=False,
                include_tables=False,
                favor_precision=True
            )

            if extracted:
                text = extracted.strip()
                status = "Текст извлечён"
            else:
                status = "Основной текст не найден"
        else:
            status = f"HTTP {response.status_code}"

    except requests.RequestException as e:
        status = f"Ошибка запроса: {type(e).__name__}"

    article_texts.append(text)
    fetch_statuses.append(status)

    print(f"{i}/{MAX_ARTICLES}: {status}")
    time.sleep(1.5)

# Добавляем результаты в датасет
df.loc[df.index[:MAX_ARTICLES], "article_text"] = article_texts
df.loc[df.index[:MAX_ARTICLES], "fetch_status"] = fetch_statuses

# Для остальных строк заполняем пустые значения
df["article_text"] = df["article_text"].fillna("")
df["fetch_status"] = df["fetch_status"].fillna("Не проверялось")

df["article_text_length"] = df["article_text"].str.len()

df.to_csv(
    "/content/digital_harassment_kz_with_text.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nИзвлечено текстов:", (df["article_text_length"] > 0).sum())
display(df[[
    "title", "source", "fetch_status", "article_text_length"
]].head(15))

1/60: Основной текст не найден


2/60: Основной текст не найден


3/60: Основной текст не найден


4/60: Основной текст не найден


5/60: Основной текст не найден


...

57/60: Основной текст не найден


58/60: Основной текст не найден


59/60: Основной текст не найден


60/60: Основной текст не найден

Извлечено текстов: 90


,title,source,fetch_status,article_text_length
3,Отец Алсу подозревается в вымогательстве двух ...,TengriNews,Основной текст не найден,0
4,Ошибки в Damumed могут привести к шантажу и вы...,TengriNews,Основной текст не найден,0
5,Избившего таксиста дипломата заподозрили в вым...,TengriNews,Основной текст не найден,0
6,"В Алматы задержан лидер ОПГ ""Атаба"" за вымогат...",TengriNews,Основной текст не найден,0
7,Руководитель облздрава Мангистауской области п...,TengriNews,Основной текст не найден,0
8,Крупную схему вымогательства денег у туристов ...,TengriNews,Основной текст не найден,0
9,Ерлан Карин об инфо-рэкете: мы не должны поощр...,TengriNews,Основной текст не найден,0
10,Новости по теме шантаж - TengriNews,TengriNews,Основной текст не найден,0
11,Второй подсудимый по делу экс-главы АРЕМ расск...,TengriNews,Основной текст не найден,0
12,Жалобы на насилие и вымогательство: управляюще...,TengriNews,Основной текст не найден,0


Ячейка 9. Интерактивный дашборд

In [9]:
import gradio as gr
import plotly.express as px
import pandas as pd

dashboard_df = df.copy()

# Подготовка значений фильтров
all_categories = sorted({
    category.strip()
    for value in dashboard_df["category"].fillna("")
    for category in str(value).split(",")
    if category.strip()
})

all_sources = sorted(
    dashboard_df["source"].fillna("").replace("", "Неизвестный источник").unique()
)

def make_dashboard(category, source, search):
    data = dashboard_df.copy()

    if category != "Все категории":
        data = data[
            data["category"].fillna("").apply(
                lambda value: category in [
                    part.strip() for part in str(value).split(",")
                ]
            )
        ]

    if source != "Все источники":
        data = data[data["source"].fillna("") == source]

    if search and search.strip():
        query = search.strip()
        mask = (
            data["title"].fillna("").str.contains(
                query, case=False, regex=False
            )
            | data["summary"].fillna("").str.contains(
                query, case=False, regex=False
            )
            | data["article_text"].fillna("").str.contains(
                query, case=False, regex=False
            )
        )
        data = data[mask]

    if data.empty:
        empty_fig = px.bar(
            x=["Нет данных"],
            y=[0],
            title="По заданным фильтрам ничего не найдено"
        )
        return 0, empty_fig, empty_fig, data

    # Если в записи несколько категорий — учитываем каждую
    category_rows = (
        data.assign(
            category=data["category"].fillna("").str.split(",")
        )
        .explode("category")
    )
    category_rows["category"] = category_rows["category"].str.strip()

    category_counts = (
        category_rows[category_rows["category"] != ""]
        .groupby("category")
        .size()
        .reset_index(name="Количество")
        .sort_values("Количество", ascending=False)
    )

    fig_categories = px.bar(
        category_counts,
        x="category",
        y="Количество",
        title="Материалы по категориям",
        labels={"category": "Категория"}
    )

    source_counts = (
        data.groupby("source")
        .size()
        .reset_index(name="Количество")
        .sort_values("Количество", ascending=False)
    )

    fig_sources = px.bar(
        source_counts,
        x="source",
        y="Количество",
        title="Материалы по источникам",
        labels={"source": "Источник"}
    )

    display_cols = [
        "date", "title", "category", "source",
        "summary", "url", "fetch_status"
    ]
    display_cols = [c for c in display_cols if c in data.columns]

    return len(data), fig_categories, fig_sources, data[display_cols]

with gr.Blocks(title="Digital Harassment KZ") as app:
    gr.Markdown("""
    # Digital Harassment KZ
    ### Анализ публикаций о шантаже и цифровом преследовании в Казахстане

    Материалы собраны из доступных новостных источников.
    Категории сформированы автоматической фильтрацией и требуют проверки.
    """)

    with gr.Row():
        category_filter = gr.Dropdown(
            ["Все категории"] + all_categories,
            value="Все категории",
            label="Категория"
        )
        source_filter = gr.Dropdown(
            ["Все источники"] + all_sources,
            value="Все источники",
            label="Источник"
        )

    search_box = gr.Textbox(
        label="Поиск",
        placeholder="Например: шантаж, угроза, кибербуллинг"
    )

    total = gr.Number(label="Количество материалов", value=len(dashboard_df))
    category_plot = gr.Plot(label="Категории")
    source_plot = gr.Plot(label="Источники")
    results_table = gr.Dataframe(
        label="Найденные публикации",
        interactive=False,
        wrap=True
    )

    controls = [category_filter, source_filter, search_box]
    outputs = [total, category_plot, source_plot, results_table]

    for control in controls:
        control.change(
            fn=make_dashboard,
            inputs=controls,
            outputs=outputs
        )

    app.load(
        fn=make_dashboard,
        inputs=controls,
        outputs=outputs
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9d6626a8996b831b9c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Ячейка 10

In [10]:
from google.colab import files

df.to_csv(
    "/content/Digital_Harassment_KZ_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

files.download("/content/Digital_Harassment_KZ_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from google.colab import files

files.download("/content/Digital_Harassment_KZ_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>